<a href="https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w6_finance_rag/llm_260417_finance_rag_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 260417 ETF RAG 고도화 -- 하이브리드 검색 + 평가 + 추천 시스템

**6주차 Day 4** - 스마트 필터 / 하이브리드 검색 / 검색 평가(Hit Rate, MRR) / MMR / LLM Reranking / Content-based Filtering

어제(d3)까지 ETF 데이터 수집 -> Knowledge Base 구축 -> FAISS 벡터 스토어 + BM25 키워드 검색 -> 메타데이터 필터링까지 했다.  
오늘은 이 검색 시스템을 **고도화**하고 **평가**하고, 마지막에 **추천 시스템** 기초까지 간다.

| 주제 | 비유 |
|------|------|
| 스마트 필터 | 고객이 "싼 거 줘"라고 하면, 점원이 알아서 가격 필터를 거는 것 |
| 하이브리드 검색 | 구글 = 키워드 + 의미 검색을 섞어서 최적 결과 |
| Normalize | 수학 100점 만점, 영어 50점 만점 -> 둘 다 0~1로 맞춰야 비교 가능 |
| Hit Rate | 10문제 중 몇 개 맞췄니? (정답률) |
| MRR | 정답이 1등으로 나오면 1점, 3등이면 1/3점 (순위도 반영) |
| MMR | 비슷한 음식만 추천하면 질리니까, 다양한 메뉴도 섞어주는 것 |
| LLM Reranking | AI 심사위원이 후보를 다시 순위 매기기 |
| Content-based Filtering | 짜장면 좋아하면 -> 짜장면 맛집 추천 (취향 기반) |

## 0. Setup

In [ ]:
!pip install -q faiss-cpu kiwipiepy langchain langchain-community langchain-core langchain-openai numpy openai pandas rank-bm25

In [ ]:
# --- Colab 전용 ---
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import numpy as np
import pandas as pd
import json
import re
from langchain_core.documents import Document

client = OpenAI()
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

## 1. ETF Knowledge Base 재구축

어제(d3)에서 실시간 데이터를 수집해서 벡터 스토어를 만들었다.  
오늘은 **고정 데이터**로 빠르게 재구축한다.  
핵심 구조: `Document(page_content=텍스트, metadata=메타정보)` -> FAISS에 저장

In [ ]:
# 12개 ETF 정보 (어제 수집한 데이터를 고정값으로 정리)
etf_knowledge_base = [
    {"name": "KODEX 200",             "category": "국내주식", "expense_ratio": 0.15,
     "return_1y":  8.5, "risk_level": "중간", "dividend_yield": 1.8, "volatility": 15.2,
     "keywords": ["코스피", "대형주", "인덱스", "분산투자", "국내주식"]},
    {"name": "KODEX 미국S&P500TR",    "category": "해외주식", "expense_ratio": 0.05,
     "return_1y": 25.3, "risk_level": "중간", "dividend_yield": 0.0, "volatility": 18.5,
     "keywords": ["미국", "S&P500", "대형주", "성장", "해외주식"]},
    {"name": "ACE 미국배당다우존스",   "category": "배당",     "expense_ratio": 0.01,
     "return_1y": 12.1, "risk_level": "낮음", "dividend_yield": 3.5, "volatility": 10.3,
     "keywords": ["미국", "배당", "배당성장", "저비용", "안정"]},
    {"name": "TIGER 2차전지테마",     "category": "테마",     "expense_ratio": 0.45,
     "return_1y":-15.2, "risk_level": "높음", "dividend_yield": 0.0, "volatility": 35.7,
     "keywords": ["2차전지", "배터리", "테마", "성장", "고위험"]},
    {"name": "KODEX 미국나스닥100TR", "category": "해외주식", "expense_ratio": 0.05,
     "return_1y": 32.1, "risk_level": "높음", "dividend_yield": 0.0, "volatility": 25.3,
     "keywords": ["미국", "나스닥", "기술주", "성장", "IT"]},
    {"name": "TIGER 미국필라델피아반도체", "category": "해외주식", "expense_ratio": 0.49,
     "return_1y": 45.0, "risk_level": "높음", "dividend_yield": 0.0, "volatility": 38.2,
     "keywords": ["반도체", "AI", "미국", "기술주", "고위험"]},
    {"name": "KODEX 골드선물(H)",     "category": "원자재",   "expense_ratio": 0.68,
     "return_1y": 18.7, "risk_level": "중간", "dividend_yield": 0.0, "volatility": 20.4,
     "keywords": ["금", "원자재", "인플레이션", "헤지", "안전자산"]},
    {"name": "TIGER 국고채10년",      "category": "채권",     "expense_ratio": 0.07,
     "return_1y":  4.2, "risk_level": "낮음", "dividend_yield": 2.8, "volatility":  5.1,
     "keywords": ["채권", "국고채", "안전", "이자", "저위험"]},
    {"name": "KODEX 단기채권PLUS",    "category": "채권",     "expense_ratio": 0.03,
     "return_1y":  3.8, "risk_level": "낮음", "dividend_yield": 3.2, "volatility":  1.5,
     "keywords": ["단기채", "안전", "예금대안", "저위험", "이자"]},
    {"name": "TIGER 리츠부동산인프라", "category": "부동산",   "expense_ratio": 0.29,
     "return_1y":  6.5, "risk_level": "중간", "dividend_yield": 4.2, "volatility": 12.8,
     "keywords": ["리츠", "부동산", "배당", "인프라", "실물자산"]},
    {"name": "KODEX 200TR",           "category": "국내주식", "expense_ratio": 0.12,
     "return_1y":  9.1, "risk_level": "중간", "dividend_yield": 0.0, "volatility": 14.9,
     "keywords": ["코스피", "대형주", "인덱스", "성장", "국내주식"]},
    {"name": "TIGER 고배당저변동",    "category": "배당",     "expense_ratio": 0.30,
     "return_1y":  7.2, "risk_level": "낮음", "dividend_yield": 5.2, "volatility":  8.7,
     "keywords": ["배당", "저변동", "안정", "배당성장", "저위험"]},
]

In [ ]:
# LLM으로 각 ETF에 대한 설명 생성 -> Document로 변환
def generate_description(etf):
    """LLM에게 ETF 정보를 주고 1~2문장 설명을 생성"""
    prompt = f"""다음 ETF의 특징을 한국어 1~2문장으로 간결히 설명하세요.
    이름 : {etf['name']}
    카테고리 : {etf['category']}
    키워드 : {','.join(etf['keywords'])}
    수수료 : {etf['expense_ratio']}% / 배당수익률: {etf['dividend_yield']}%
    1년수익률 : {etf['return_1y']}% / 변동성: {etf['volatility']}%
    """
    return llm.invoke([{'role': 'user', 'content': prompt}]).content.strip()

# Document 리스트 생성
documents = []
for etf in etf_knowledge_base:
    desc = generate_description(etf)
    text = (f"{etf['name']} ({etf['category']}): {desc} "
            f"키워드: {', '.join(etf['keywords'])} "
            f"수수료 {etf['expense_ratio']}%, 배당수익률: {etf['dividend_yield']}% "
            f"수익률: {etf['return_1y']}% / 변동성: {etf['volatility']}%")
    metadata = {k: v for k, v in etf.items() if k != 'keywords'}
    metadata['keywords'] = ', '.join(etf['keywords'])
    documents.append(Document(page_content=text, metadata=metadata))

print(f"총 {len(documents)}개 Document 생성")
print(documents[0].page_content[:120])

In [ ]:
# FAISS 벡터 스토어 생성
vectorstore = LangchainFAISS.from_documents(documents, embeddings)
print(f"벡터 스토어 문서 수: {vectorstore.index.ntotal}")

In [ ]:
# BM25 키워드 검색 설정 (Kiwi 한국어 토크나이저 사용)
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi

kiwi = Kiwi()

def kiwi_tokenize(text):
    """한국어 형태소 분석 -> 명사/영어/숫자만 추출
    비유: 문장에서 '의미 있는 단어'만 골라내는 것"""
    tokens = kiwi.tokenize(text)
    result = []
    for token in tokens:
        if token.tag in ("SL", "SN", "NNG", "NNP"):  # 영어, 숫자, 일반명사, 고유명사
            result.append(token.form.lower())
    return result

# 전체 문서를 토큰화 -> BM25 인덱스 생성
corpus = [kiwi_tokenize(doc.page_content) for doc in documents]
bm25 = BM25Okapi(corpus)

def bm25_search(query, k=5):
    """BM25 키워드 검색: TF-IDF 기반 점수로 문서 순위 매기기"""
    tokens = kiwi_tokenize(query)
    scores = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:k]
    return [(documents[i], scores[i]) for i in top_idx if scores[i] > 0]

# 테스트
for doc, score in bm25_search("배당 ETF", k=3):
    print(f"  {doc.metadata['name']} | score: {score:.2f}")

## 2. 메타데이터 필터 검색 (복습)

어제(d3) 만든 함수.  
벡터 검색으로 후보를 넓게 가져온 뒤, 메타데이터 조건으로 걸러내는 방식.

```
비유: 서점에서 "프로그래밍"으로 검색 -> 20권 나옴 -> 그 중 가격 2만원 이하만 필터
```

In [ ]:
def filtered_search(vectorstore, query, filters=None, k=5, fetch_k=20):
    """벡터 검색 + 메타데이터 필터링
    filters 예시: {"category": "해외주식", "expense_ratio": {"less_than": 0.1}}"""
    results = vectorstore.similarity_search_with_score(query, fetch_k)
    if not filters:
        return results[:k]
    
    filtered = []
    for doc, score in results:
        match = True
        for key, condition in filters.items():
            val = doc.metadata.get(key)
            if isinstance(condition, dict):
                # {"less_than": 0.1} 또는 {"greater_than": 3.0} 형태
                if "less_than" in condition and val > condition["less_than"]:
                    match = False
                if "greater_than" in condition and val < condition["greater_than"]:
                    match = False
            elif val != condition:  # 정확히 일치하는지 비교
                match = False
        if match:
            filtered.append((doc, score))
    
    return filtered[:k] if filtered else results[:k]  # 필터 결과 없으면 전체에서 k개

# 테스트: 해외주식 + 수수료 0.1% 이하
results = filtered_search(vectorstore, "저비용 해외 ETF",
                          filters={"category": "해외주식", "expense_ratio": {"less_than": 0.1}})
for doc, score in results:
    print(f"  {doc.metadata['name']} | 수수료 {doc.metadata['expense_ratio']}%")

## 3. 스마트 필터 -- LLM이 자연어에서 필터 자동 추출

사용자가 "수수료 0.1% 이하이면서 배당 3% 이상인 ETF" 라고 말하면,  
LLM이 자동으로 `{"expense_ratio": {"less_than": 0.1}, "dividend_yield": {"greater_than": 3.0}}`을 만들어준다.

```
비유: 고객이 "좀 싸고 배당 좋은 거" -> 점원이 알아서 조건표를 만들어서 검색
                                      (사람이 직접 필터 안 써도 됨!)
```

**주의**: LLM이 JSON을 마크다운으로 감싸서 줄 때가 있다 (```json ... ```).  
이 예외처리가 실무에서 매우 중요하다!

In [ ]:
def smart_filtered_search(query):
    """자연어 쿼리에서 메타데이터 필터를 자동 추출
    핵심: LLM에게 가능한 필터 필드와 형식을 알려주고, JSON으로 응답하게 함"""
    prompt = f"""사용자 쿼리에서 ETF 검색 필터를 추출하세요.
    
    쿼리 : {query}
    
    사용 가능한 필터 필드:
    - category : 국내주식, 해외주식, 배당, 테마, 원자재, 채권, 부동산
    - risk_level : 낮음, 중간, 높음
    - expense_ratio : 숫자 (% "less_than"/"greater_than")
    - dividend_yield : 숫자 (%)
    - return_1y : 숫자 (%)
    - volatility : 숫자 (%)
    
    JSON 형식으로만 응답하세요. 필터가 없으면 {{}}.
    예: {{"category":"해외주식","expense_ratio":{{"less_than":0.5}}}}
    """
    
    response = llm.invoke([{'role': 'user', 'content': prompt}]).content
    try:
        # 마크다운 코드블록 감싸기 예외처리
        # 비유: LLM이 답을 예쁘게 포장해서 줄 때가 있는데, 포장지를 벗겨야 한다
        if "```" in response:
            response = response.split("```")[1].replace("json", "")
        return json.loads(response)
    except Exception:
        return {}

# 테스트
test_query = "수수료 0.1% 이하이면서 배당 3% 이상인 ETF"
filters = smart_filtered_search(test_query)
print(f"쿼리: {test_query}")
print(f"추출된 필터: {json.dumps(filters, ensure_ascii=False)}")

In [ ]:
def smart_document_search(query, k=3):
    """스마트 필터 + 벡터 검색을 결합한 최종 검색 함수
    1) LLM이 쿼리에서 필터 추출
    2) 필터 적용해서 벡터 검색
    3) 결과 없으면 필터 없이 재검색"""
    filters = smart_filtered_search(query)
    print(f"쿼리: {query}")
    print(f"필터: {json.dumps(filters, ensure_ascii=False)}")
    
    results = filtered_search(vectorstore, query, filters=filters, k=k)
    if not results:
        print("필터 조건에 맞는 결과가 없습니다. 필터 없이 재검색...")
        results = filtered_search(vectorstore, query, k=k)
    
    print(f"결과: {len(results)}개")
    for doc, score in results:
        m = doc.metadata
        print(f"  {m['name']} | {m['category']} | 수수료 {m['expense_ratio']}% | 배당 {m['dividend_yield']}%")
    return results

# 테스트
smart_document_search("위험 낮고 배당 2% 이상인 ETF")

## 4. 하이브리드 검색 -- 벡터 + 키워드를 합치기

벡터 검색과 키워드(BM25) 검색은 각각 장단점이 있다:

| 검색 방법 | 장점 | 단점 |
|----------|------|------|
| 벡터 검색 | 의미적 유사성 ("안전한 투자" -> 채권 찾음) | 키워드 정확 매칭 약함 |
| BM25 키워드 | 정확한 키워드 매칭 ("KODEX 200" -> 정확히 찾음) | 동의어/의미 파악 못함 |

**하이브리드 검색**: 두 결과를 `alpha` 가중치로 합치는 것

```
최종 점수 = alpha * 벡터점수(정규화) + (1-alpha) * BM25점수(정규화)
```

### 왜 Normalize가 필요한가?

벡터 검색은 **거리 기반** (0.3, 0.5, 1.2...), BM25는 **점수 기반** (2.1, 5.3, 8.7...).  
스케일이 다르니까 그냥 더하면 BM25 쪽으로 치우친다.  

```
비유: 수학 100점 만점에서 80점, 영어 50점 만점에서 40점
      그냥 더하면 수학이 높지만, %로 바꾸면 영어가 더 높다
      -> Min-Max 정규화로 둘 다 0~1 범위로 맞춰야 공정한 비교
```

In [ ]:
def hybrid_search(query, alpha=0.5, k=5, filters=None):
    """하이브리드 검색: 벡터 + BM25를 가중합
    alpha=1.0이면 벡터만, alpha=0.0이면 BM25만 사용"""
    
    # 1) 벡터 검색 (거리 -> 유사도로 변환: 1/(1+거리))
    vec_results = vectorstore.similarity_search_with_score(query, k=20)
    vec_scores = {}
    for doc, dist in vec_results:
        vec_scores[doc.metadata['name']] = 1 / (1 + dist)  # 거리의 역수 = 유사도
    
    # 2) BM25 키워드 검색
    bm25_results = bm25_search(query, k=20)
    bm25_scores = {}
    for doc, score in bm25_results:
        bm25_scores[doc.metadata['name']] = score
    
    # 3) Min-Max 정규화: 둘 다 0~1 범위로 맞추기
    def normalize(scores):
        if not scores:
            return scores
        vals = list(scores.values())
        min_, max_ = min(vals), max(vals)
        rng = max_ - min_ if max_ != min_ else 1.0  # 전부 같은 값이면 0으로 나누기 방지
        return {k: (v - min_) / rng for k, v in scores.items()}
    
    vec_norm = normalize(vec_scores)
    bm25_norm = normalize(bm25_scores)
    
    # 4) 두 결과의 합집합에서 가중합 계산
    all_names = set(vec_norm) | set(bm25_norm)
    combined = {}
    for name in all_names:
        v = vec_norm.get(name, 0)   # 벡터 검색에 없으면 0
        b = bm25_norm.get(name, 0)  # BM25에 없으면 0
        combined[name] = alpha * v + (1 - alpha) * b
    
    # 5) 필터 적용 (선택사항)
    if filters:
        name_to_doc = {d.metadata['name']: d for d in documents}
        filtered = {}
        for name, score in combined.items():
            doc = name_to_doc.get(name)
            if not doc:
                continue
            match = True
            for key, condition in filters.items():
                val = doc.metadata.get(key)
                if isinstance(condition, dict):
                    if "less_than" in condition and val > condition["less_than"]:
                        match = False
                    if "greater_than" in condition and val < condition["greater_than"]:
                        match = False
                elif val != condition:
                    match = False
            if match:
                filtered[name] = score
        combined = filtered
    
    # 6) 점수 기준 정렬 -> 상위 k개 반환
    sorted_results = sorted(combined.items(), key=lambda x: x[1], reverse=True)
    return sorted_results[:k]

In [ ]:
# alpha 값에 따른 검색 결과 비교
# alpha=0.0: BM25만, alpha=0.5: 반반, alpha=1.0: 벡터만
query = "미국 기술 성장 ETF"
for alpha in [0.0, 0.5, 1.0]:
    results = hybrid_search(query, alpha=alpha, k=3)
    names = [name for name, _ in results]
    print(f"alpha={alpha:.1f}: {names}")

## 5. 검색 품질 평가 -- Hit Rate & MRR

검색 시스템을 만들었으면, **얼마나 잘 찾는지** 측정해야 한다.  
정답지(evaluation data)를 만들어놓고, 검색 결과와 비교하는 방식.

### Hit Rate (적중률)
```
비유: 시험 10문제 중 7개 맞추면 적중률 70%
      검색이 정답 문서를 "하나라도" 찾으면 hit!
```

### MRR (Mean Reciprocal Rank)
```
비유: 정답이 1등으로 나오면 1점, 2등이면 0.5점, 3등이면 0.33점
      순위가 높을수록 점수가 높다 -> "빨리 찾는 것"을 보상
      
Hit Rate:  정답을 찾았느냐 안 찾았느냐 (이진)
MRR:       정답을 몇 등으로 찾았느냐 (순위 반영)
```

In [ ]:
# 평가용 데이터: 쿼리별 정답 문서 리스트
# 비유: 시험 문제 + 정답지
eval_queries = [
    {"query": "안전한 배당 ETF", "relevant": ["ACE 미국배당다우존스", "TIGER 국고채10년", "KODEX 단기채권PLUS"]},
    {"query": "미국 기술주 투자", "relevant": ["KODEX 미국나스닥100TR", "TIGER 미국필라델피아반도체"]},
    {"query": "2차전지 관련 ETF", "relevant": ["TIGER 2차전지테마"]},
    {"query": "금 투자 인플레이션 방어", "relevant": ["KODEX 골드선물(H)"]},
    {"query": "국내 대형주 인덱스", "relevant": ["KODEX 200"]},
    {"query": "부동산 배당 투자", "relevant": ["TIGER 리츠부동산인프라"]},
    {"query": "저비용 미국 ETF", "relevant": ["KODEX 미국S&P500TR", "ACE 미국배당다우존스"]},
    {"query": "채권 안정 수익", "relevant": ["TIGER 국고채10년", "KODEX 단기채권PLUS"]},
]

In [ ]:
def hit_rate(eval_data, search_fn, k=5):
    """Hit Rate: 정답 문서를 하나라도 찾으면 hit
    비유: 범인을 용의자 목록에 넣었으면 성공"""
    hits = 0
    for item in eval_data:
        results = search_fn(item['query'], k=k)
        # search_fn 결과가 (name, score) 튜플일 수도, Document일 수도 있으므로 처리
        found = [r[0] if isinstance(r, tuple) else r.metadata.get('name', '') for r in results]
        if any(rel in found for rel in item['relevant']):
            hits += 1
    return hits / len(eval_data)


def mrr(eval_data, search_fn, k=5):
    """MRR (Mean Reciprocal Rank): 정답의 순위에 역수를 취해 평균
    비유: 1등으로 맞추면 1점, 3등이면 1/3점 -> 빨리 찾을수록 좋은 점수"""
    rr_sum = 0
    for item in eval_data:
        results = search_fn(item['query'], k=k)
        found = [r[0] if isinstance(r, tuple) else r.metadata.get('name', '') for r in results]
        for rank, name in enumerate(found, 1):  # 1부터 시작
            if name in item['relevant']:
                rr_sum += 1 / rank  # 1등이면 1, 2등이면 0.5, 3등이면 0.33...
                break  # 첫 번째 정답만 반영
    return rr_sum / len(eval_data)

In [ ]:
# 각 검색 방법을 평가 함수에 넣을 수 있는 형태로 래핑
# (search_fn은 query와 k를 받아서 (name, score) 튜플 리스트를 반환해야 함)

def vec_fn(q, k=5):
    results = vectorstore.similarity_search(q, k=k)
    return [(doc.metadata['name'], 1.0) for doc in results]

def bm25_fn(q, k=5):
    results = bm25_search(q, k=k)
    return [(doc.metadata['name'], score) for doc, score in results]

def hybrid_fn(q, k=5):
    return hybrid_search(q, alpha=0.5, k=k)

In [ ]:
# 3가지 검색 방법의 Hit Rate & MRR 비교
print(f"{'방법':<10} | {'HR@3':>6} | {'HR@5':>6} | {'MRR@3':>7} | {'MRR@5':>7}")
print("-" * 52)
for name, fn in [("vector", vec_fn), ("bm25", bm25_fn), ("hybrid", hybrid_fn)]:
    hr3 = hit_rate(eval_queries, fn, k=3)
    hr5 = hit_rate(eval_queries, fn, k=5)
    mrr3 = mrr(eval_queries, fn, k=3)
    mrr5 = mrr(eval_queries, fn, k=5)
    print(f"{name:<10} | {hr3:>6.3f} | {hr5:>6.3f} | {mrr3:>7.4f} | {mrr5:>7.4f}")

> **k값 선택 기준**: 모바일 앱처럼 화면이 작으면 k=3 (사용자가 적게 봄).  
> 데스크탑 검색처럼 스크롤 많이 하면 k=5~10으로 늘릴 수 있다.

## 6. MMR -- 검색 결과의 다양성 확보

벡터 검색은 유사한 문서만 모아오는 경향이 있다.  
예: "좋은 투자 상품 추천" -> 비슷비슷한 해외주식 ETF만 5개 나옴

**MMR (Maximum Marginal Relevance)**는 **유사성 + 다양성**을 동시에 고려한다.

```
MMR = lambda * similarity(쿼리, 문서) + (1-lambda) * diversity(문서끼리 얼마나 다른지)

lambda=1.0 -> 벡터 검색과 동일 (유사한 것만)
lambda=0.0 -> 다양한 것만 (관련성 무시)
lambda=0.5 -> 유사하면서도 다양한 것 (균형)
```

비유: 뷔페에서 음식 고를 때  
- similarity만: 좋아하는 초밥만 5접시  
- MMR: 초밥 2접시 + 스테이크 1접시 + 샐러드 1접시 + 디저트 1접시

In [ ]:
# LangChain FAISS에 내장된 MMR 함수 사용
query = "좋은 투자 상품 추천"

# 일반 벡터 검색
sim_results = vectorstore.similarity_search(query, k=5)
# MMR 검색: fetch_k=12에서 후보 뽑고, 그 중 k=5개를 다양하게 선택
mmr_results = vectorstore.max_marginal_relevance_search(
    query, k=5, fetch_k=12, lambda_mult=0.5
)

print("=== 일반 벡터 검색 (비슷한 것만) ===")
for doc in sim_results:
    print(f"  {doc.metadata['name']} | {doc.metadata['category']}")

print("\n=== MMR 검색 (다양성 확보) ===")
for doc in mmr_results:
    print(f"  {doc.metadata['name']} | {doc.metadata['category']}")

> **lambda_mult 값에 따른 차이**:  
> - `lambda_mult=1.0`: similarity_search와 동일  
> - `lambda_mult=0.0`: 최대한 다양한 문서만 선택  
> - `lambda_mult=0.5`: 균형 (보통 이 값을 기본으로 사용)

## 7. LLM Reranking -- AI가 검색 결과 순위를 재조정

검색(Retriever)이 가져온 후보를, **LLM이 다시 읽고 순위를 매기는** 것.

```
Retriever -> 8개 후보 -> [LLM Reranker] -> 상위 5개만 Generator에 전달
```

비유: 1차 서류 전형(검색)으로 8명 뽑고 -> 면접관(LLM)이 다시 보고 5명 선발

**장점**: LLM이 쿼리 맥락을 이해하고 정교하게 판단  
**단점**: LLM 호출 비용 & 시간 (그래서 후보를 적게 줘야 효율적)

In [ ]:
def llm_rerank(query, candidates, k=5):
    """LLM이 후보 ETF를 보고 쿼리에 적합한 순서로 재정렬
    candidates: [(name, score), ...] 형태의 하이브리드 검색 결과"""
    name_to_doc = {d.metadata['name']: d for d in documents}
    
    # 후보 정보를 텍스트로 정리
    cand_text = ""
    for i, (name, score) in enumerate(candidates):
        doc = name_to_doc.get(name)
        if doc:
            m = doc.metadata
            cand_text += (f"{i+1}. {name} ({m['category']}) | "
                         f"수수료 {m['expense_ratio']}% | 수익률 {m['return_1y']}% | "
                         f"배당 {m['dividend_yield']}%\n")
    
    # LLM에게 순위 매기기 요청
    prompt = f"""사용자 쿼리에 가장 적합한 ETF를 순위대로 정렬하세요.
    
    쿼리 : {query}
    후보 ETF : {cand_text}
    가장 적합한 {k}개를 순위대로 번호만 응답하세요.
    예 : 3, 1, 5, 2, 4"""
    
    response = llm.invoke([{"role": "user", "content": prompt}]).content
    # 정규표현식으로 숫자만 추출
    # 비유: LLM 답변에서 번호표만 뽑아내는 것
    numbers = [int(x) for x in re.findall(r'\d+', response)]
    
    # 중복 제거하면서 순서 유지
    reranked = []
    seen = set()
    for n in numbers:
        if 1 <= n <= len(candidates) and n not in seen:
            reranked.append(candidates[n - 1])
            seen.add(n)
    
    # LLM이 빠뜨린 후보는 뒤에 추가 (안전장치)
    for c in candidates:
        if c not in reranked:
            reranked.append(c)
    
    return reranked[:k]

In [ ]:
# 테스트: 하이브리드 검색 -> LLM Reranking
query = "초보자에게 안전한 ETF"
initial = hybrid_search(query, alpha=0.5, k=8)  # 넓게 8개 검색
reranked = llm_rerank(query, initial, k=5)       # LLM이 5개로 압축

print("=== 하이브리드 검색 결과 (초기 8개) ===")
for name, score in initial:
    print(f"  {name}: {score:.3f}")

print("\n=== LLM Reranking 후 (상위 5개) ===")
for name, score in reranked:
    print(f"  {name}: {score:.3f}")

## 8. Content-based Filtering -- 콘텐츠 기반 추천

지금까지는 **검색** (사용자가 질문 -> 문서 찾기)이었다.  
이제 **추천** (사용자 취향 -> 비슷한 상품 추천)을 해본다.

### 추천 시스템의 2가지 방법

| 방법 | 원리 | 비유 |
|------|------|------|
| **Content-based** | 상품 특성이 비슷한 걸 추천 | 짜장면 좋아하면 -> 짜장면 맛집 추천 |
| **Collaborative** | 비슷한 사용자가 산 걸 추천 | 나와 취향 비슷한 친구가 산 걸 추천 |

### Collaborative Filtering의 현실적 어려움
```
           상품1  상품2  상품3  ...  상품1억
사용자A     O                         
사용자B            O      O
사용자C     O             O
...                                  
사용자3만

-> 1억 x 3만 행렬인데 대부분 비어있음 (Sparse Matrix)
-> 행렬 분해(Matrix Factorization)로 차원 축소해서 사용
```

오늘은 **Content-based Filtering**만 구현한다.

### 8-1. ETF를 벡터로 변환

추천을 하려면 ETF끼리의 유사도를 구해야 한다.  
그러려면 각 ETF의 특성을 **숫자 벡터**로 바꿔야 한다.

```
ETF = [리스크, 수익률, 배당률, 수수료, 변동성]
     = [0.33,  0.69,  0.30,  0.15,  0.38]   <- 5차원 벡터
```

**주의: Normalize가 필수!**  
변동성은 0~40 범위, 수수료는 0~1 범위 -> 그냥 벡터로 만들면 변동성이 방향을 지배한다.  
각 feature를 최대값으로 나눠서 비슷한 범위로 맞춰준다.

In [ ]:
# 추천용 데이터 (위 etf_knowledge_base와 동일)
etf_data = [
    {"name": "KODEX 200",             "category": "국내주식", "expense_ratio": 0.15, "return_1y":  8.5, "risk_level": "중간", "dividend_yield": 1.8, "volatility": 15.2},
    {"name": "KODEX 미국S&P500TR",    "category": "해외주식", "expense_ratio": 0.05, "return_1y": 25.3, "risk_level": "중간", "dividend_yield": 0.0, "volatility": 18.5},
    {"name": "ACE 미국배당다우존스",   "category": "배당",     "expense_ratio": 0.01, "return_1y": 12.1, "risk_level": "낮음", "dividend_yield": 3.5, "volatility": 10.3},
    {"name": "TIGER 2차전지테마",     "category": "테마",     "expense_ratio": 0.45, "return_1y":-15.2, "risk_level": "높음", "dividend_yield": 0.0, "volatility": 35.7},
    {"name": "KODEX 미국나스닥100TR", "category": "해외주식", "expense_ratio": 0.05, "return_1y": 32.1, "risk_level": "높음", "dividend_yield": 0.0, "volatility": 25.3},
    {"name": "TIGER 미국필라델피아반도체", "category": "섹터","expense_ratio": 0.49, "return_1y": 45.0, "risk_level": "높음", "dividend_yield": 0.0, "volatility": 38.2},
    {"name": "KODEX 골드선물(H)",     "category": "원자재",   "expense_ratio": 0.68, "return_1y": 18.7, "risk_level": "중간", "dividend_yield": 0.0, "volatility": 20.4},
    {"name": "TIGER 국고채10년",      "category": "채권",     "expense_ratio": 0.07, "return_1y":  4.2, "risk_level": "낮음", "dividend_yield": 2.8, "volatility":  5.1},
    {"name": "KODEX 단기채권PLUS",    "category": "채권",     "expense_ratio": 0.03, "return_1y":  3.8, "risk_level": "낮음", "dividend_yield": 3.2, "volatility":  1.5},
    {"name": "TIGER 리츠부동산인프라", "category": "리츠",     "expense_ratio": 0.29, "return_1y":  6.5, "risk_level": "중간", "dividend_yield": 4.8, "volatility": 12.8},
    {"name": "KODEX 200TR",           "category": "국내주식", "expense_ratio": 0.12, "return_1y":  9.1, "risk_level": "중간", "dividend_yield": 0.0, "volatility": 14.9},
    {"name": "TIGER 고배당저변동",    "category": "배당",     "expense_ratio": 0.30, "return_1y":  7.2, "risk_level": "낮음", "dividend_yield": 5.2, "volatility":  8.7},
]

etf_df = pd.DataFrame(etf_data)
etf_df.head()

In [ ]:
# 리스크 레벨을 숫자로 매핑
risk_map = {"낮음": 1, "중간": 2, "높음": 3}

def etf_to_vector(etf):
    """ETF 특성을 5차원 벡터로 변환 (각 feature를 0~1 범위로 정규화)
    비유: 사람의 키/몸무게/나이를 같은 단위로 맞추는 것"""
    return np.array([
        risk_map[etf['risk_level']] / 3,       # 1~3 -> 0.33~1.0
        (etf['return_1y'] + 30) / 80,           # -15~45 범위 -> 0~1 근처
        etf['dividend_yield'] / 6,              # 0~5.2 -> 0~0.87
        etf['expense_ratio'],                   # 이미 0~0.68 범위
        etf['volatility'] / 40                  # 0~38 -> 0~0.95
    ])

# 전체 ETF 벡터화
item_vectors = np.array([etf_to_vector(e) for e in etf_data])
item_names = [e['name'] for e in etf_data]

print(f"벡터 shape: {item_vectors.shape}")  # (12, 5) = 12개 ETF, 5차원
print(f"KODEX 200 벡터: {item_vectors[0]}")

In [ ]:
# 코사인 유사도 함수
from numpy.linalg import norm

def cosine_sim(a, b):
    """두 벡터의 코사인 유사도 (방향이 비슷하면 1에 가까움)
    비유: 두 화살표가 같은 방향을 가리키면 유사"""
    return float(np.dot(a, b) / (norm(a) * norm(b) + 1e-8))

### 8-2. 아이템 유사도 행렬 구축

12개 ETF끼리의 유사도를 미리 다 계산해놓는 것.

```
         KODEX200  SP500  배당...
KODEX200   1.00    0.93   0.85
SP500      0.93    1.00   0.78
배당...    0.85    0.78   1.00

대각선은 자기 자신 = 항상 1.0
대칭 행렬: sim(A,B) = sim(B,A)
```

In [ ]:
def build_item_similarity(vectors):
    """모든 아이템 쌍의 코사인 유사도 행렬 생성"""
    n = len(vectors)
    sim = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            sim[i, j] = cosine_sim(vectors[i], vectors[j])
    return sim

item_sim = build_item_similarity(item_vectors)
print(f"유사도 행렬 shape: {item_sim.shape}")
print(f"KODEX 200 <-> 자기자신: {item_sim[0, 0]:.4f}")  # 1.0
print(f"KODEX 200 <-> KODEX 200TR: {item_sim[0, 10]:.4f}")  # 비슷한 상품이니 높을 것

### 8-3. 유사 아이템 추천

In [ ]:
def cbf_similar_items(target_name, top_k=5):
    """콘텐츠 기반 추천: target과 유사한 ETF top_k개 반환
    비유: 짜장면 좋아하면 -> 짜장면과 비슷한 음식 추천"""
    idx = item_names.index(target_name)
    # 자기 자신 제외하고 유사도 순으로 정렬
    scores = [(item_names[j], item_sim[idx, j]) for j in range(len(item_names)) if j != idx]
    return sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

# 테스트: KODEX 200과 비슷한 ETF는?
print("KODEX 200과 유사한 ETF:")
for name, score in cbf_similar_items('KODEX 200', top_k=5):
    print(f"  {score:.4f} | {name}")

### 8-4. 카테고리 다양성을 가진 추천

유사도만으로 추천하면 같은 카테고리만 나올 수 있다.  
(예: 국내주식 ETF 추천하면 국내주식만 5개...)  

**카테고리가 겹치지 않게** 추천하면 더 유용한 결과가 나온다.

```
비유: 여행지 추천에서 "서울, 부산, 대구, 대전, 광주" (전부 한국) 보다
      "서울, 도쿄, 파리, 뉴욕, 방콕" (다양한 나라)이 더 넓은 선택지
```

In [ ]:
def cbf_diverse(target_name, top_k=5):
    """카테고리 다양성을 보장하는 추천
    유사도 순으로 보되, 이미 나온 카테고리는 건너뛰기"""
    idx = item_names.index(target_name)
    scores = [(j, item_sim[idx, j]) for j in range(len(item_names)) if j != idx]
    scores.sort(key=lambda x: x[1], reverse=True)
    
    result, seen_cats = [], set()
    for j, score in scores:
        cat = etf_data[j]['category']
        if cat in seen_cats:  # 이미 이 카테고리에서 뽑았으면 건너뛰기
            continue
        result.append((item_names[j], round(score, 3), cat))
        seen_cats.add(cat)
        if len(result) >= top_k:
            break
    return result

# 테스트: KODEX 200 기준 다양한 카테고리 추천
print("KODEX 200 기준 다양한 추천:")
for name, score, cat in cbf_diverse('KODEX 200', top_k=5):
    print(f"  {score:.3f} | {cat:<8} | {name}")

## 정리

### 오늘 다룬 내용 한눈에

| 단계 | 기능 | 핵심 |
|------|------|------|
| 스마트 필터 | 자연어 -> JSON 필터 자동 추출 | LLM이 점원 역할 |
| 하이브리드 검색 | 벡터 + BM25 가중합 | Normalize 필수! |
| 평가 (Hit Rate) | 정답을 찾았는지 | 이진 판단 |
| 평가 (MRR) | 정답을 몇 등으로 찾았는지 | 순위 반영 |
| MMR | 다양한 문서 검색 | lambda로 유사성/다양성 조절 |
| LLM Reranking | 후보를 LLM이 재정렬 | 비용 vs 정확도 trade-off |
| Content-based | 상품 특성 벡터로 유사 상품 추천 | 벡터 정규화 중요 |
| 다양성 추천 | 카테고리 중복 없이 추천 | seen_cats으로 필터 |

### 빠진 주차 복습 포인트 (w4~w5 연결)

- **w4**: Hit Rate / MRR 개념 처음 등장 -> 오늘 ETF 데이터에 적용  
- **w5**: Reranking (pointwise, pairwise, listwise) -> 오늘 LLM reranking 실전 적용  
- **w5**: Score Filtering / AdaptiveFilter -> 필요하면 hybrid_search 결과에 적용 가능